# 🗺️ Módulo 4: Clustering de Vulnerabilidad Provincial en Piura
## Proyecto: AgroSmart Perú — Universidad / Software Inteligente

**Objetivo:** Agrupar y zonificar las 8 provincias de Piura (*Piura, Sullana, Morropón, Sechura, Paita, Talara, Huancabamba, Ayabaca*) según su exposición a lluvias torrenciales, déficit de drenaje y pérdidas históricas mediante **K-Means**.  
**Justificación Matemática:** Selección óptima de $k=3$ mediante el **Método del Codo (Inertia)** y el **Coeficiente de Silueta**.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
os.makedirs('../models', exist_ok=True)
print("Librerías importadas correctamente.")


### 1. Carga de Datos Territoriales de las 8 Provincias de Piura


In [ ]:
df_prov = pd.read_csv('../data/provincias_piura_riesgo.csv')
features = ['lluvia_max_historica_mm', 'pct_area_inundable', 'perdida_cosecha_nino_pct', 'deficit_drenaje_pct']

print(f"Dimensiones: {df_prov.shape[0]} provincias x {df_prov.shape[1]} columnas")
df_prov[['provincia', 'valle_principal', 'cultivo_predominante'] + features]


### 2. Estandarización y Justificación del Número de Clusters (K)


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_prov[features])

k_range = range(2, 6)
inercia = []
siluetas = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercia.append(km.inertia_)
    siluetas.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(k_range, inercia, 'bo-', lw=2)
ax1.set_xlabel('Número de Clusters (k)')
ax1.set_ylabel('Inercia (WCSS)')
ax1.set_title('Método del Codo')

ax2.plot(k_range, siluetas, 'ro-', lw=2)
ax2.set_xlabel('Número de Clusters (k)')
ax2.set_ylabel('Coeficiente de Silueta')
ax2.set_title('Evaluación de Silueta')

plt.tight_layout()
plt.show()

print(f"🏆 Mejor puntuación de Silueta obtenida con K=3: {siluetas[1]:.3f}")


### 3. Entrenamiento con K=3 y Reducción Dimensional PCA


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_prov['cluster'] = kmeans.fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df_prov['pca_1'] = pca_coords[:, 0]
df_prov['pca_2'] = pca_coords[:, 1]

# Mapeo según el orden de pérdida promedio
resumen_perdida = df_prov.groupby('cluster')['perdida_cosecha_nino_pct'].mean().sort_values(ascending=False)
orden = resumen_perdida.index.tolist()

etiquetas = {orden[0]: 'Vulnerabilidad Extrema', orden[1]: 'Vulnerabilidad Alta', orden[2]: 'Vulnerabilidad Moderada'}
colores = {orden[0]: '#d9534f', orden[1]: '#f0ad4e', orden[2]: '#5cb85c'}

df_prov['nivel_vulnerabilidad'] = df_prov['cluster'].map(etiquetas)
df_prov[['provincia', 'cultivo_predominante', 'perdida_cosecha_nino_pct', 'nivel_vulnerabilidad']]


### 4. Gráfico de Dispersión PCA (Zonificación Provincial)


In [ ]:
plt.figure(figsize=(9, 6))
for c_id, name in etiquetas.items():
    sub = df_prov[df_prov['cluster'] == c_id]
    plt.scatter(sub['pca_1'], sub['pca_2'], s=220, label=name, color=colores[c_id], edgecolors='k')
    for _, row in sub.iterrows():
        plt.annotate(row['provincia'], (row['pca_1'] + 0.08, row['pca_2'] + 0.08), fontsize=11, fontweight='bold')

plt.xlabel('Componente Principal 1 (Exposición y Lluvias Extremas)')
plt.ylabel('Componente Principal 2 (Déficit de Drenaje)')
plt.title(f'Zonificación de Provincias de Piura por Clusters de Riesgo (Silhouette = {silhouette_score(X_scaled, kmeans.labels_):.3f})')
plt.legend()
plt.tight_layout()
plt.show()


### 5. Exportación del Modelo


In [ ]:
joblib.dump({'kmeans': kmeans, 'scaler': scaler, 'pca': pca}, '../models/kmeans_provincias.pkl')
print("✅ Modelo K-Means exportado exitosamente a '../models/kmeans_provincias.pkl'")
